### TRANSFERIR LOS DATOS A LA CAPA SILVER
**IMPORTAMOS LAS LIBRERIAS**

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

**EXTRAEMOS LOS DATOS DE LA CAPA BRONZE**

In [0]:
# OBTENEMOS LOS DATOS DE LA CAPA BRONZE
silver_bronze= spark.table("spotify_catalog.bronze.spotify_tracks")

display(silver_bronze.limit(5))

**TRATAMIENTO DE LOS DATOS**

In [0]:
# ELIMINAMOS LOS REGISTROS CON DATOS NULOS
silver_clean = silver_bronze.filter(col("track_id").isNotNull())

silver_clean = silver_clean.filter(col("track_name").isNotNull() & (trim(col("track_name")) != ""))

In [0]:
# CONVERTIMOS LA DURACIÓN DE LOS TRACKS A MINUTOS
silver_clean = silver_clean.withColumn("duration_minutes",round(col("duration_ms") / 60000, 2))

In [0]:
# VERIFICAMOS SI LAS FECHAS DE LANZAMIENTO SON DIAS, MESES O AÑOS
silver_clean = silver_clean.withColumn(
    "release_date_precision",
    when(length(col("release_date")) == 4, "year")
    .when(length(col("release_date")) == 7, "month")
    .when(length(col("release_date")) == 10, "day")
    .otherwise("unknown")
)

silver_clean["release_date_precision"].value_counts()

In [0]:
# NORMALIZAMOS LAS FECHAS DE LANZAMIENTO COLOCANDO TODOS EN FORMATO DE DIA
silver_clean = silver_clean.withColumn(
    "release_date_clean",
    when(
        length(col("release_date")) == 10,
        to_date(col("release_date"), "yyyy-MM-dd")
    )
    .when(
        length(col("release_date")) == 7,
        to_date(
            concat(col("release_date"), lit("-01")),
            "yyyy-MM-dd"
        )
    )
    .when(
        length(col("release_date")) == 4,
        to_date(
            concat(col("release_date"), lit("-01-01")),
            "yyyy-MM-dd"
        )
    )
    .otherwise(None)
)

display(silver_clean.limit(20))

In [0]:
# CATEGORIZAMOS LA POPULARIDAD DE LOS TRACKS
silver_clean = silver_clean.filter((col("popularity")>=0) & (col("popularity")<=100)
                ).withColumn(
                    "popularity_level",  
                    when(col("popularity") <= 25, "Baja")
                    .when(col("popularity") <= 50, "Media")
                    .when(col("popularity") <= 75, "alta")
                    .when(col("popularity") <= 100, "Muy alta")
                    .otherwise("Desconocida")
                )

display(silver_clean.limit(10))

In [0]:
# ELIMINAMOS REGISTROS DUPLICADOS MANTENIENDO EL REGISTRO CON FECHA DE EXTRACCION MAS RECENTE

window = Window.partitionBy("track_id").orderBy(col("extraction_timestamp").desc())

silver_clean_final = (
    silver_clean
    .withColumn(
        "rn",
        row_number().over(window)
    )
    .filter(col("rn") == 1)
    .drop("rn")
)

In [0]:
# COMPARAMOS LA CANTIDAD DE REGISTROS ENTRE LA CAPA BRONZE Y SILVER

print("Cantidad de Registros en Capa Bronze: ", silver_bronze.count())
print("Cantidad de Registros en Capa Silver: ", silver_clean_final.count())

**GUARDAMOS LOS DATOS EN LA CAPA SILVER**

In [0]:
(
    silver_clean_final
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "spotify_catalog.silver.spotify_tracks"
    )
)